# Synthetische Bauteile – Inspektion

Prüft ein einzelnes synthetisches Bauteil vor der Pipeline-Auswertung.

## Methodik

1. **Referenz** (unverändertes CAD) laden – dort ist der Y=0-Split
   eindeutig, weil noch keine Transformation drin ist. Ergibt `ref_A` und `ref_B`.
2. **Erwartete Synthetik rekonstruieren** – Ground-Truth-Transformation
   auf `ref_B` anwenden: `expected = ref_A + T(ref_B)`.
3. **A/B-Zuordnung in der tatsächlichen Synthetik** über KDTree-Vergleich
   zu `ref_A` und `T(ref_B)` – funktioniert unabhängig von der Transformation.
4. **Quantitative Prüfung** – wenn der Generator korrekt arbeitet, muss
   jeder synthetische Punkt einen exakten Match in `expected` haben.

## Konvention

`ty_mm > 0` in der CSV bedeutet: der Spalt öffnet sich um `|ty_mm|`.
Der Generator invertiert `ty` beim Anwenden auf Werkstück B (das im
negativen Y-Bereich sitzt). Diese Inversion wird hier bei der
Rekonstruktion ebenfalls angewendet.


## Konfiguration

In [ ]:
from pathlib import Path
import numpy as np
import open3d as o3d
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

# Zu inspizierendes Bauteil (ohne .ply-Endung)
CASE_NAME = "T_Y_+01.500mm"

# Pfade
SYNTH_DIR = Path("../data/raw/synthetic_scans")
CAD_CACHE_DIR = Path("../data/outputs/cad/Baugruppe_Beispielteil_V-Naht_1.5mm_Spalt")

# Farb-Konventionen (identisch zur Pipeline)
COLOR_A = "#1976D2"     # blau
COLOR_B = "#E53935"     # rot
COLOR_REF = "#B0BEC5"   # hellgrau für Overlay


## 1. Ground-Truth-Metadaten

In [ ]:
df = pd.read_csv(SYNTH_DIR / "synthetic_metadata.csv")
row = df[df["filename"] == f"{CASE_NAME}.ply"]

if row.empty:
    raise ValueError(f"Kein Eintrag für {CASE_NAME}.ply in synthetic_metadata.csv")

r = row.iloc[0]
tx, ty, tz = r["tx_mm"], r["ty_mm"], r["tz_mm"]
rx, ry, rz = r["rx_deg"], r["ry_deg"], r["rz_deg"]

print(f"Kategorie:     {r['category']}")
print(f"Beschreibung:  {r['description']}")
print()
print(f"Ground Truth (User-Konvention: +ty = Spalt öffnet sich):")
print(f"  Translation: Δx={tx:+.4f} mm  Δy={ty:+.4f} mm  Δz={tz:+.4f} mm")
print(f"  Rotation:    rx={rx:+.4f}°   ry={ry:+.4f}°   rz={rz:+.4f}°")

## 2. Referenz aufbereiten

CAD laden, top-surface-Filter anwenden (wie im Generator), dann bei Y=0
in A und B trennen.

In [ ]:
pcd_ref_full = o3d.io.read_point_cloud(str(CAD_CACHE_DIR / "pointcloud.ply"))
normals = np.asarray(pcd_ref_full.normals)
top_mask = normals[:, 2] > 0.5
ref = np.asarray(pcd_ref_full.points)[top_mask]

ref_a_mask = ref[:, 1] >= 0
ref_a = ref[ref_a_mask]
ref_b = ref[~ref_a_mask]

print(f"Referenz gesamt (top-surface):  {len(ref):,} Punkte")
print(f"  Werkstück A (Y ≥ 0):          {len(ref_a):,} Punkte")
print(f"  Werkstück B (Y < 0):          {len(ref_b):,} Punkte")
print(f"  Y-Bereiche: A ∈ [{ref_a[:,1].min():+.3f}, {ref_a[:,1].max():+.3f}] | "
      f"B ∈ [{ref_b[:,1].min():+.3f}, {ref_b[:,1].max():+.3f}]")

## 3. Erwartete Synthetik rekonstruieren

Ground-Truth-Transformation auf `ref_B` anwenden. **Wichtig:** `ty` wird
invertiert (User-Konvention: +ty = Spaltöffnung, Werkstück B wandert
tatsächlich in -Y). Konvention identisch zum Generator.

In [ ]:
def build_transformation(tx, ty, tz, rx_deg, ry_deg, rz_deg):
    rx_rad, ry_rad, rz_rad = np.deg2rad([rx_deg, ry_deg, rz_deg])
    Rx = np.array([[1, 0, 0],
                   [0, np.cos(rx_rad), -np.sin(rx_rad)],
                   [0, np.sin(rx_rad),  np.cos(rx_rad)]])
    Ry = np.array([[ np.cos(ry_rad), 0, np.sin(ry_rad)],
                   [ 0,              1, 0],
                   [-np.sin(ry_rad), 0, np.cos(ry_rad)]])
    Rz = np.array([[np.cos(rz_rad), -np.sin(rz_rad), 0],
                   [np.sin(rz_rad),  np.cos(rz_rad), 0],
                   [0,               0,              1]])
    R = Rx @ Ry @ Rz
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = [tx, ty, tz]
    return T

# ty invertieren, wie im Generator
T_gt = build_transformation(tx, -ty, tz, rx, ry, rz)
ref_b_transformed = (T_gt[:3, :3] @ ref_b.T).T + T_gt[:3, 3]

expected = np.vstack([ref_a, ref_b_transformed])

print(f"Erwartete Synthetik: {len(expected):,} Punkte")
print(f"  aus ref_A ({len(ref_a):,}) + T(ref_B) ({len(ref_b_transformed):,})")
print(f"  Y-Bereich: [{expected[:,1].min():+.3f}, {expected[:,1].max():+.3f}]")

## 4. Tatsächliche Synthetik laden

In [ ]:
pcd_synth = o3d.io.read_point_cloud(str(SYNTH_DIR / f"{CASE_NAME}.ply"))
synth = np.asarray(pcd_synth.points)

print(f"Tatsächliche Synthetik: {len(synth):,} Punkte")
print(f"  Y-Bereich: [{synth[:,1].min():+.3f}, {synth[:,1].max():+.3f}]")
print(f"  Match zur erwarteten Punktzahl: "
      f"{'✓' if len(synth) == len(expected) else '✗ ABWEICHUNG'}")

## 5. Quantitative Prüfung

Jeder Punkt in der synthetischen Wolke muss einen nahezu perfekten Match
in `expected` haben. Mittlere Distanz nahe 0 = Generator arbeitet korrekt.

In [ ]:
tree_expected = cKDTree(expected)
d_synth_to_expected, _ = tree_expected.query(synth, k=1)

print(f"Distanz synthetische Punkte → erwartete Punkte:")
print(f"  mean = {d_synth_to_expected.mean():.6f} mm")
print(f"  max  = {d_synth_to_expected.max():.6f} mm")
print(f"  p95  = {np.percentile(d_synth_to_expected, 95):.6f} mm")

if d_synth_to_expected.max() < 0.01:
    print("\n✓ Generator arbeitet korrekt – synthetische Wolke entspricht "
          "der Rekonstruktion aus Ground Truth.")
else:
    print("\n⚠ Signifikante Abweichungen – Generator produziert nicht das "
          "erwartete Ergebnis.")

## 6. A/B-Zuordnung in der Synthetik

Für jeden Punkt: Distanz zum nächsten `ref_A`-Punkt vs. zum nächsten
`T(ref_B)`-Punkt. Der kleinere gewinnt.

In [ ]:
tree_a = cKDTree(ref_a)
tree_b = cKDTree(ref_b_transformed)

d_a, _ = tree_a.query(synth, k=1)
d_b, _ = tree_b.query(synth, k=1)

is_a = d_a <= d_b
n_a_synth = int(is_a.sum())
n_b_synth = int((~is_a).sum())

print(f"Werkstück A in Synthetik: {n_a_synth:,} Punkte")
print(f"Werkstück B in Synthetik: {n_b_synth:,} Punkte")
print(f"Match zur Referenz-Punktzahl:  A {'✓' if n_a_synth == len(ref_a) else '✗'}  "
      f"B {'✓' if n_b_synth == len(ref_b) else '✗'}")

## 7. Visualisierung Gesamtbauteil

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), dpi=150)

ax = axes[0]
ax.scatter(synth[is_a, 0], synth[is_a, 1],
           c=COLOR_A, s=1, alpha=0.5, label=f"Werkstück A ({n_a_synth:,})")
ax.scatter(synth[~is_a, 0], synth[~is_a, 1],
           c=COLOR_B, s=1, alpha=0.5, label=f"Werkstück B ({n_b_synth:,})")
ax.axhline(0, color="gray", linestyle=":", alpha=0.4)
ax.set_xlabel("X (mm)")
ax.set_ylabel("Y (mm)")
ax.set_aspect("equal")
ax.set_title(f"Draufsicht (XY) – {CASE_NAME}")
ax.legend(markerscale=5)
ax.grid(alpha=0.3)

ax = axes[1]
ax.scatter(synth[is_a, 1], synth[is_a, 2],
           c=COLOR_A, s=2, alpha=0.5, label="Werkstück A")
ax.scatter(synth[~is_a, 1], synth[~is_a, 2],
           c=COLOR_B, s=2, alpha=0.5, label="Werkstück B")
ax.axvline(0, color="gray", linestyle=":", alpha=0.4)
ax.set_xlabel("Y (mm)")
ax.set_ylabel("Z (mm)")
ax.set_aspect("equal")
ax.set_title(f"Seitenansicht (YZ) – {CASE_NAME}")
ax.legend(markerscale=5)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
synth_a = synth[is_a]
synth_b = synth[~is_a]
print(f"Werkstück A: Y-min = {synth_a[:,1].min():+.4f} mm  (erwartet: +0.75)")
print(f"Werkstück B: Y-max = {synth_b[:,1].max():+.4f} mm  (bei +1.5 Öffnung: -2.25)")
print(f"Tatsächliche Spaltbreite: {synth_a[:,1].min() - synth_b[:,1].max():.4f} mm")

## 8. Detail Nahtbereich

Zoom auf den Y-Bereich [-5, +5], damit der Spalt sichtbar wird.
Bei 1.5 mm Soll-Spalt und `T_Y_+01.500mm` müsste ein Spalt von 3 mm
zwischen Werkstück A und Werkstück B zu sehen sein.

In [ ]:
detail_mask = (synth[:, 1] > -5) & (synth[:, 1] < 5)

fig, ax = plt.subplots(figsize=(10, 8), dpi=150)
ax.scatter(synth[detail_mask & is_a, 0], synth[detail_mask & is_a, 1],
           c=COLOR_A, s=3, alpha=0.6, label="Werkstück A")
ax.scatter(synth[detail_mask & ~is_a, 0], synth[detail_mask & ~is_a, 1],
           c=COLOR_B, s=3, alpha=0.6, label="Werkstück B")
ax.axhline(0, color="gray", linestyle=":", alpha=0.4, label="Y = 0")
ax.set_xlabel("X (mm)")
ax.set_ylabel("Y (mm)")
ax.set_title(f"Detail Nahtbereich – {CASE_NAME}")
ax.legend(markerscale=5)
ax.grid(alpha=0.3)
ax.set_ylim(-5, 5)
plt.tight_layout()
plt.show()

## 9. Overlay mit Referenz

Referenz grau im Hintergrund, transformierte Synthetik in Farbe.
Werkstück A muss deckungsgleich mit der Referenz liegen, Werkstück B
zeigt die eingebrachte Transformation.

In [ ]:
rng = np.random.default_rng(0)
def _sub(pts, n=40_000):
    if len(pts) <= n:
        return pts, np.arange(len(pts))
    idx = rng.choice(len(pts), n, replace=False)
    return pts[idx], idx

ref_plot, _ = _sub(ref)
synth_plot, synth_idx = _sub(synth)
is_a_plot = is_a[synth_idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 5), dpi=150)

ax = axes[0]
ax.scatter(ref_plot[:, 0], ref_plot[:, 1],
           c=COLOR_REF, s=1, alpha=0.4, label="Referenz")
ax.scatter(synth_plot[is_a_plot, 0], synth_plot[is_a_plot, 1],
           c=COLOR_A, s=2, alpha=0.6, label="Werkstück A")
ax.scatter(synth_plot[~is_a_plot, 0], synth_plot[~is_a_plot, 1],
           c=COLOR_B, s=2, alpha=0.6, label="Werkstück B")
ax.set_xlabel("X (mm)")
ax.set_ylabel("Y (mm)")
ax.set_aspect("equal")
ax.set_title(f"Overlay Draufsicht – {CASE_NAME}")
ax.legend(markerscale=5, framealpha=0.95)
ax.grid(alpha=0.3)

ax = axes[1]
ax.scatter(ref_plot[:, 1], ref_plot[:, 2],
           c=COLOR_REF, s=2, alpha=0.4, label="Referenz")
ax.scatter(synth_plot[is_a_plot, 1], synth_plot[is_a_plot, 2],
           c=COLOR_A, s=2, alpha=0.6, label="Werkstück A")
ax.scatter(synth_plot[~is_a_plot, 1], synth_plot[~is_a_plot, 2],
           c=COLOR_B, s=2, alpha=0.6, label="Werkstück B")
ax.set_xlabel("Y (mm)")
ax.set_ylabel("Z (mm)")
ax.set_aspect("equal")
ax.set_title(f"Overlay Seitenansicht – {CASE_NAME}")
ax.legend(markerscale=5, framealpha=0.95)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Querschnitt (optional)

Y-Z-Schnitt eines schmalen X-Streifens. Nützlich um Rotationen um X und
Öffnungswinkel im Detail zu sehen.

In [ ]:
X_MIN, X_MAX = -10.0, 10.0
cs_mask_ref = (ref[:, 0] >= X_MIN) & (ref[:, 0] < X_MAX)
cs_mask_synth = (synth[:, 0] >= X_MIN) & (synth[:, 0] < X_MAX)

fig, ax = plt.subplots(figsize=(9, 6), dpi=150)
ax.scatter(ref[cs_mask_ref, 1], ref[cs_mask_ref, 2],
           c=COLOR_REF, s=8, alpha=0.5, label="Referenz")
ax.scatter(synth[cs_mask_synth & is_a, 1], synth[cs_mask_synth & is_a, 2],
           c=COLOR_A, s=10, alpha=0.7, label="Werkstück A")
ax.scatter(synth[cs_mask_synth & ~is_a, 1], synth[cs_mask_synth & ~is_a, 2],
           c=COLOR_B, s=10, alpha=0.7, label="Werkstück B")
ax.axhline(0, color="gray", linestyle=":", alpha=0.5)
ax.axvline(0, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("Y (mm)")
ax.set_ylabel("Z (mm)")
ax.set_aspect("equal")
ax.set_title(f"Querschnitt X ∈ [{X_MIN:+.1f}, {X_MAX:+.1f}] – {CASE_NAME}")
ax.legend(framealpha=0.95)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()